In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.table import Table

In [ ]:
import arya

In [ ]:
from desi_utils.plotting import plot_wrapped_spectrum

In [ ]:
arya.init("mnras")

In [ ]:
filename = "data/sky_spectra_vac_v1.fits"

In [ ]:
with fits.open(filename) as f:
    wavelength = f["WAVELENGTH"].data
    flux = f["FLUX"].data
    spec_info = Table.read(f[-1])

In [ ]:
# Identify indices for different conditions
twilight_idxs = (
    (spec_info['SUNALT'] > -20) & 
                     (spec_info['MOONALT'] <= -5) &
                     (spec_info['SUNSEP'] <= 110)
)

moon_idxs = (
    (spec_info['SUNALT'] <= -20) &
                 (spec_info['MOONALT'] > 5) &
                 (spec_info['MOONSEP'] <= 90) &
                 (spec_info['MOONFRAC'] > 0.5)
)

dark_idxs = (
    (spec_info['SUNALT'] <= -20) & 
    (spec_info['MOONALT'] <= -5)
)


In [ ]:
moon_flux = flux[moon_idxs, :]
moon_flux /= np.median(moon_flux, axis=1).reshape((-1, 1))


dark_flux = flux[dark_idxs, :]
dark_flux /= np.median(dark_flux, axis=1).reshape((-1, 1))

In [ ]:
med_moon_flux = np.median(moon_flux, axis=0)
low_moon_flux = np.quantile(moon_flux, 0.16, axis=0)
high_moon_flux = np.quantile(moon_flux, 0.84, axis=0)



med_dark_flux = np.median(dark_flux, axis=0)
low_dark_flux = np.quantile(dark_flux, 0.16, axis=0)
high_dark_flux = np.quantile(dark_flux, 0.84, axis=0)

In [ ]:
spec = {
    # "pca": (wavelength, pca.components_[0] / np.quantile(pca.components_[0], 0.99)),
    "mean": (wavelength, med_moon_flux),
    "dark": (wavelength, med_dark_flux * 0.5),
}

plot_wrapped_spectrum(spec, ylim=(0.1, 50), yscale="log")

# Save the spectra

In [ ]:
spec_table = Table({
    "flux": med_dark_flux,
    "wavelength": wavelength,
}
                  )

In [ ]:
spec_table.write("data/skyspectrum.fits")

In [ ]:
spec = {
    # "pca": (wavelength, pca.components_[0] / np.quantile(pca.components_[0], 0.99)),
    "dark": (wavelength, med_dark_flux * 0.5),
    "filter": (wavelength, med_dark_flux > 5),
}

plot_wrapped_spectrum(spec, ylim=(0.1, 50), yscale="log")

# more plots

In [ ]:
spectra = {
    "median": (wavelength, med_dark_flux),
    "low": (wavelength, low_dark_flux),
    "high": (wavelength, high_dark_flux)
}

plot_wrapped_spectrum(spectra,  ylim=(0.1, 3))

In [ ]:
spectra = {
    "median": (wavelength, med_moon_flux),
    "low": (wavelength, low_moon_flux),
    "high": (wavelength, high_moon_flux)
}

plot_wrapped_spectrum(spectra,  ylim=(0.1, 3))

In [ ]:
for idx in range(100):
    plt.plot(wavelength, moon_flux[idx, :], color="black", alpha=0.01, lw=0.3)

plt.yscale("log")
plt.xlabel("wavelength / angstrom")
plt.ylabel("flux")

# SKLearn

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(5)
pca.fit(moon_flux)

In [ ]:
plt.plot(np.cumsum(pca.explained_variance_ratio_))

In [ ]:
for comp in pca.components_[0:1]:
    plt.plot(wavelength, comp)